In [1]:
# !pip install pandas numpy scikit-learn joblib matplotlib seaborn
import json, joblib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

SEED = 42
DATA = Path("../data2/IoT Network Intrusion Dataset.csv")
OUT  = Path("processed"); OUT.mkdir(exist_ok=True)

In [2]:
df = pd.read_csv(DATA)
print(df.shape)
print(df["Label"].value_counts())
print(df["Cat"].value_counts())
df.head()

(625783, 86)
Label
Anomaly    585710
Normal      40073
Name: count, dtype: int64
Cat
Mirai                415677
Scan                  75265
DoS                   59391
Normal                40073
MITM ARP Spoofing     35377
Name: count, dtype: int64


,Flow_ID,Src_IP,Src_Port,Dst_IP,Dst_Port,Protocol,Timestamp,Flow_Duration,Tot_Fwd_Pkts,Tot_Bwd_Pkts,...,Active_Std,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min,Label,Cat,Sub_Cat
0,192.168.0.13-192.168.0.16-10000-10101-17,192.168.0.13,10000,192.168.0.16,10101,17,25/07/2019 03:25:53 AM,75,1,1,...,0.0,0.0,0.0,75.0,0.000000,75.0,75.0,Anomaly,Mirai,Mirai-Ackflooding
1,192.168.0.13-222.160.179.132-554-2179-6,222.160.179.132,2179,192.168.0.13,554,6,26/05/2019 10:11:06 PM,5310,1,2,...,0.0,0.0,0.0,2655.0,2261.327486,4254.0,1056.0,Anomaly,DoS,DoS-Synflooding
2,192.168.0.13-192.168.0.16-9020-52727-6,192.168.0.16,52727,192.168.0.13,9020,6,11/07/2019 01:24:48 AM,141,0,3,...,0.0,0.0,0.0,70.5,0.707107,71.0,70.0,Anomaly,Scan,Scan Port OS
3,192.168.0.13-192.168.0.16-9020-52964-6,192.168.0.16,52964,192.168.0.13,9020,6,04/09/2019 03:58:17 AM,151,0,2,...,0.0,0.0,0.0,151.0,0.000000,151.0,151.0,Anomaly,Mirai,Mirai-Hostbruteforceg
4,192.168.0.1-239.255.255.250-36763-1900-17,192.168.0.1,36763,239.255.255.250,1900,17,10/09/2019 01:41:18 AM,153,2,1,...,0.0,0.0,0.0,76.5,0.707107,77.0,76.0,Anomaly,Mirai,Mirai-Hostbruteforceg


In [3]:
ID_COLS = ["Flow_ID", "Src_IP", "Dst_IP", "Timestamp", "Src_Port"]
LABELS  = ["Label", "Cat", "Sub_Cat"]

# 1) inf -> NaN, then drop those rows (only 368 rows)
df = df.replace([np.inf, -np.inf], np.nan)
print("rows with NaN/inf:", df.isna().any(axis=1).sum())
df = df.dropna()

# 2) drop identifier columns
df = df.drop(columns=ID_COLS)

# 3) drop constant columns
feat_cols = [c for c in df.columns if c not in LABELS]
const = [c for c in feat_cols if df[c].nunique() <= 1]
print("constant columns dropped:", const)
df = df.drop(columns=const)

# 4) drop exact duplicates BEFORE splitting
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"duplicates removed: {before - len(df)}  ->  {len(df)} rows left")

print(df["Label"].value_counts(normalize=True).round(3))

rows with NaN/inf: 368
constant columns dropped: ['Fwd_PSH_Flags', 'Fwd_URG_Flags', 'Fwd_Byts/b_Avg', 'Fwd_Pkts/b_Avg', 'Fwd_Blk_Rate_Avg', 'Bwd_Byts/b_Avg', 'Bwd_Pkts/b_Avg', 'Bwd_Blk_Rate_Avg', 'Init_Fwd_Win_Byts', 'Fwd_Seg_Size_Min']
duplicates removed: 424970  ->  200445 rows left
Label
Anomaly    0.864
Normal     0.136
Name: proportion, dtype: float64


In [4]:
y_bin = (df["Label"] == "Anomaly").astype(int)   # 1 = attack, 0 = normal
y_cat = df["Cat"]                                 # kept for a multiclass option later
X = df.drop(columns=LABELS)
feature_names = X.columns.tolist()
print(X.shape, y_bin.mean().round(3))

(200445, 68) 0.864


In [5]:
X_tr, X_tmp, y_tr, y_tmp, c_tr, c_tmp = train_test_split(
    X, y_bin, y_cat, test_size=0.30, stratify=y_bin, random_state=SEED)
X_val, X_te, y_val, y_te, c_val, c_te = train_test_split(
    X_tmp, y_tmp, c_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

for n, y in [("train", y_tr), ("val", y_val), ("test", y_te)]:
    print(n, len(y), "attack share:", round(y.mean(), 3))

train 140311 attack share: 0.864
val 30067 attack share: 0.864
test 30067 attack share: 0.864


In [6]:
skew = X_tr.skew()
# only log-transform columns that are non-negative in train and strongly skewed
log_cols = [c for c in feature_names if X_tr[c].min() >= 0 and skew[c] > 2]
print(len(log_cols), "columns log1p-transformed")

def apply_log(d):
    d = d.copy()
    d[log_cols] = np.log1p(d[log_cols].clip(lower=0))
    return d

X_tr_l, X_val_l, X_te_l = apply_log(X_tr), apply_log(X_val), apply_log(X_te)

51 columns log1p-transformed


In [7]:
scaler = MinMaxScaler()
X_tr_s  = scaler.fit_transform(X_tr_l)
X_val_s = scaler.transform(X_val_l)
X_te_s  = scaler.transform(X_te_l)

# values can fall outside [0,1] on val/test; clip so epsilon stays meaningful later
X_val_s = np.clip(X_val_s, 0, 1)
X_te_s  = np.clip(X_te_s, 0, 1)
print(X_tr_s.min(), X_tr_s.max())

0.0 1.0


In [8]:
np.save(OUT/"X_train.npy", X_tr_s); np.save(OUT/"y_train.npy", y_tr.values)
np.save(OUT/"X_val.npy",   X_val_s); np.save(OUT/"y_val.npy",   y_val.values)
np.save(OUT/"X_test.npy",  X_te_s);  np.save(OUT/"y_test.npy",  y_te.values)
c_tr.to_csv(OUT/"cat_train.csv", index=False)
c_val.to_csv(OUT/"cat_val.csv", index=False)
c_te.to_csv(OUT/"cat_test.csv", index=False)

joblib.dump(scaler, OUT/"scaler.joblib")
json.dump({"features": feature_names, "log_cols": log_cols},
          open(OUT/"meta.json", "w"), indent=2)
print("saved to", OUT.resolve())

saved to C:\Users\arvin\Desktop\projects\adversarial-ids\balancing2\processed
